# Energy Sector Pairs Trading — Backtest Notebook

Statistical arbitrage across three energy sub-sectors using a 60-day rolling OLS hedge ratio.

**Pairs:** SLB/HAL (oilfield services) · PSX/VLO (refining) · XOM/CVX (integrated majors)  
**Formation:** 2024-03-01 → 2025-03-01  
**Backtest:** 2025-03-01 → present  

---

## Section 1 — Imports & Parameters

All strategy parameters are defined here. Adjust ENTRY_Z, STOP_Z, or ROLLING_WINDOW
to experiment with different configurations without touching the rest of the notebook.

In [64]:
import numpy as np
import pandas as pd
import shinybroker as sb
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)

PAIRS = [
    ('SLB', 'HAL', 'SLB/HAL'),
    ('PSX', 'VLO', 'PSX/VLO'),
    ('XOM', 'CVX', 'XOM/CVX'),
]

FORMATION_START = '2024-03-01'
FORMATION_END   = '2025-03-01'
BACKTEST_START  = '2025-03-01'

ENTRY_Z         =  2.0
EXIT_Z          =  0.0
STOP_Z          =  4.0

TOTAL_CAPITAL   = 100_000
PAIR_CAPITAL    = TOTAL_CAPITAL / len(PAIRS)   # 33333 per pair
COST_BPS        = 5

## Section 2 — Data Fetching

Pulls daily closing prices for all 6 tickers via IBKR/ShinyBroker.
Make sure TWS or Gateway is running on port 7497 before executing.

In [65]:
HOST      = '127.0.0.1'
PORT      = 7497        # TWS paper trading
CLIENT_ID = 9999

def make_contract(sym):
    return sb.Contract({
        'symbol': sym,
        'secType': 'STK',
        'exchange': 'SMART',
        'currency': 'USD'
    })

def fetch_daily(sym, duration='3 Y'):
    df = sb.fetch_historical_data(
        contract=make_contract(sym),
        durationStr=duration,
        barSizeSetting='1 day',
        whatToShow='Trades',
        useRTH=True,
        host=HOST,
        port=PORT,
        client_id=CLIENT_ID,
        timeout=60,
    )['hst_dta']
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df.set_index('timestamp')[['close']].rename(columns={'close': sym})

tickers = list(set(t for a, b, _ in PAIRS for t in [a, b]))
print(f'Fetching {len(tickers)} tickers...')

raw = {}
for sym in tickers:
    raw[sym] = fetch_daily(sym)
    print(f'  v {sym}')

all_prices = pd.concat(raw.values(), axis=1).dropna()
print(f'\n{len(all_prices)} trading days  ({all_prices.index[0].date()} to {all_prices.index[-1].date()})')
all_prices.head()

Fetching 6 tickers...
  v SLB
  v PSX
  v XOM
  v CVX
  v VLO
  v HAL

752 trading days  (2023-05-03 to 2026-05-01)


,SLB,PSX,XOM,CVX,VLO,HAL
timestamp,,,,,,
2023-05-03,45.2700,95.9500,107.9300,156.8300,107.0600,29.1500
2023-05-04,45.0700,92.3100,106.0400,156.2200,104.3100,29.0200
2023-05-05,45.7500,93.4400,108.6800,160.2100,107.0400,29.8800
2023-05-08,46.6100,93.9600,109.1100,159.5800,107.3800,29.9500
2023-05-09,47.1700,93.9800,109.1400,159.1200,108.4000,30.1000


## Section 3 — Pair Validation

Runs Engle-Granger cointegration and ADF tests on the formation period only.
Parameters estimated here (alpha, hedge ratio, spread mean/std) are frozen
and carried forward into the backtest — no look-ahead.

In [66]:
formation = all_prices[FORMATION_START:FORMATION_END].copy()
print(f'Formation: {len(formation)} days  ({formation.index[0].date()} to {formation.index[-1].date()})')
print()

validation = {}

for sym_a, sym_b, name in PAIRS:
    log_a = np.log(formation[sym_a])
    log_b = np.log(formation[sym_b])

    corr = log_a.corr(log_b)
    eg_stat, eg_pval, eg_crits = coint(log_a, log_b)

    ols         = OLS(log_a, add_constant(log_b)).fit()
    alpha       = ols.params.iloc[0]
    hedge_ratio = ols.params.iloc[1]
    spread      = ols.resid

    adf_stat, adf_pval, *_ = adfuller(spread, autolag='AIC')

    validation[name] = {
        'alpha':       alpha,
        'hedge_ratio': hedge_ratio,
        'spread_mean': float(spread.mean()),
        'spread_std':  float(spread.std()),
        'eg_pval':     eg_pval,
        'adf_pval':    adf_pval,
    }

    eg_flag  = 'PASS' if eg_pval  < 0.10 else 'MARGINAL'
    adf_flag = 'PASS' if adf_pval < 0.05 else 'FAIL'

    print(f'{name}')
    print(f'  Correlation  : {corr:.4f}')
    print(f'  EG p-value   : {eg_pval:.4f}  {eg_flag}')
    print(f'  ADF p-value  : {adf_pval:.4f}  {adf_flag}')
    print(f'  Hedge ratio β: {hedge_ratio:.4f}')
    print(f'  R²           : {ols.rsquared:.4f}')
    print()

Formation: 250 days  (2024-03-01 to 2025-02-28)

SLB/HAL
  Correlation  : 0.9246
  EG p-value   : 0.4486  MARGINAL
  ADF p-value  : 0.2257  FAIL
  Hedge ratio β: 0.6683
  R²           : 0.8549

PSX/VLO
  Correlation  : 0.9332
  EG p-value   : 0.1399  MARGINAL
  ADF p-value  : 0.0476  PASS
  Hedge ratio β: 0.9269
  R²           : 0.8708

XOM/CVX
  Correlation  : 0.3125
  EG p-value   : 0.3072  MARGINAL
  ADF p-value  : 0.1323  FAIL
  Hedge ratio β: 0.3059
  R²           : 0.0977



## Section 4 — Rolling Hedge Ratio

Computes a 60-day rolling OLS hedge ratio for each pair across the full dataset.
This replaces the static formation-period estimate with one that adapts over time.

In [67]:
# Rolling OLS hedge ratio — recalculate every day using a 60-day lookback window
# This replaces the static OLS estimate with a dynamic one that adapts over time

ROLLING_WINDOW = 60  # days

def rolling_hedge_ratio(prices, sym_a, sym_b, window):
    """
    For each day in prices, estimate hedge ratio using the past `window` days.
    Falls back to full-sample OLS for early rows where window isn't full yet.
    """
    log_a = np.log(prices[sym_a])
    log_b = np.log(prices[sym_b])

    hedge_ratios = []
    alphas       = []

    for i in range(len(prices)):
        start      = max(0, i - window + 1)
        window_a   = log_a.iloc[start:i+1]
        window_b   = log_b.iloc[start:i+1]

        if len(window_a) < 5:
            hr    = validation[name]['hedge_ratio']
            alpha = validation[name]['alpha']
        else:
            ols   = OLS(window_a, add_constant(window_b)).fit()
            hr    = ols.params.iloc[1]
            alpha = ols.params.iloc[0]

        hedge_ratios.append(hr)
        alphas.append(alpha)

    return np.array(hedge_ratios), np.array(alphas)


# Pre-compute rolling hedge ratios for each pair over the full dataset
# (formation + backtest combined, so indices line up cleanly)
rolling_params = {}
for sym_a, sym_b, name in PAIRS:
    hrs, alphas = rolling_hedge_ratio(all_prices, sym_a, sym_b, ROLLING_WINDOW)
    rolling_params[name] = {
        'hedge_ratios': hrs,
        'alphas':       alphas,
        'dates':        all_prices.index,
    }
    print(f'{name}: hedge ratio range = [{hrs.min():.4f}, {hrs.max():.4f}]')

SLB/HAL: hedge ratio range = [0.0915, 1.3130]
PSX/VLO: hedge ratio range = [0.0581, 1.7738]
XOM/CVX: hedge ratio range = [-0.1768, 2.1019]


## Section 5 — Backtest

Simulates daily trading signals using the rolling hedge ratio and
formation-period spread stats for z-score normalization.
Generates per-pair blotters and ledgers.

In [68]:
backtest_prices = all_prices[BACKTEST_START:].copy()
print(f'Backtest: {len(backtest_prices)} days  ({backtest_prices.index[0].date()} to {backtest_prices.index[-1].date()})')

all_blotters = []
all_ledgers  = []

for sym_a, sym_b, name in PAIRS:
    params = rolling_params[name]

    position     = 0
    trades       = []
    entry_date   = None
    entry_z      = None
    entry_prices = None

    spreads  = []
    zscores  = []
    hrs_bt   = []

    # Use formation-period spread stats for z-score normalization (no look-ahead)
    spread_mean = validation[name]['spread_mean']
    spread_std  = validation[name]['spread_std']

    for date, row in backtest_prices.iterrows():
        # Get rolling hedge ratio for this date
        idx    = list(all_prices.index).index(date)
        hr     = params['hedge_ratios'][idx]
        alpha  = params['alphas'][idx]

        log_a  = np.log(row[sym_a])
        log_b  = np.log(row[sym_b])
        spread = log_a - hr * log_b - alpha
        z      = (spread - spread_mean) / spread_std

        spreads.append(spread)
        zscores.append(z)
        hrs_bt.append(hr)

        pA = row[sym_a]
        pB = row[sym_b]

        # Stop-loss
        if position != 0 and abs(z) >= STOP_Z:
            trades.append({
                'pair':          name,
                'open_date':     entry_date,
                'close_date':    date,
                'direction':     'long_spread' if position == 1 else 'short_spread',
                'entry_z':       entry_z,
                'exit_z':        z,
                'exit_type':     'stop_loss',
                'price_A_open':  entry_prices[sym_a],
                'price_B_open':  entry_prices[sym_b],
                'price_A_close': pA,
                'price_B_close': pB,
            })
            position = 0
            continue

        # Exit
        if position == 1 and z >= EXIT_Z:
            trades.append({
                'pair':          name,
                'open_date':     entry_date,
                'close_date':    date,
                'direction':     'long_spread',
                'entry_z':       entry_z,
                'exit_z':        z,
                'exit_type':     'mean_reversion',
                'price_A_open':  entry_prices[sym_a],
                'price_B_open':  entry_prices[sym_b],
                'price_A_close': pA,
                'price_B_close': pB,
            })
            position = 0

        elif position == -1 and z <= EXIT_Z:
            trades.append({
                'pair':          name,
                'open_date':     entry_date,
                'close_date':    date,
                'direction':     'short_spread',
                'entry_z':       entry_z,
                'exit_z':        z,
                'exit_type':     'mean_reversion',
                'price_A_open':  entry_prices[sym_a],
                'price_B_open':  entry_prices[sym_b],
                'price_A_close': pA,
                'price_B_close': pB,
            })
            position = 0

        # Entry
        if position == 0:
            if z < -ENTRY_Z:
                position     = 1
                entry_date   = date
                entry_z      = z
                entry_prices = {sym_a: pA, sym_b: pB}
            elif z > ENTRY_Z:
                position     = -1
                entry_date   = date
                entry_z      = z
                entry_prices = {sym_a: pA, sym_b: pB}

    # Force-close at end of backtest
    if position != 0:
        last = backtest_prices.iloc[-1]
        trades.append({
            'pair':          name,
            'open_date':     entry_date,
            'close_date':    backtest_prices.index[-1],
            'direction':     'long_spread' if position == 1 else 'short_spread',
            'entry_z':       entry_z,
            'exit_z':        zscores[-1],
            'exit_type':     'end_of_backtest',
            'price_A_open':  entry_prices[sym_a],
            'price_B_open':  entry_prices[sym_b],
            'price_A_close': last[sym_a],
            'price_B_close': last[sym_b],
        })

    blotter = pd.DataFrame(trades)
    if len(blotter) > 0:
        blotter['open_date']  = pd.to_datetime(blotter['open_date'])
        blotter['close_date'] = pd.to_datetime(blotter['close_date'])
        blotter['hold_days']  = (blotter['close_date'] - blotter['open_date']).dt.days

        ret_A = blotter['price_A_close'] / blotter['price_A_open'] - 1
        ret_B = blotter['price_B_close'] / blotter['price_B_open'] - 1
        sign  = blotter['direction'].map({'long_spread': 1, 'short_spread': -1})

        alloc                   = PAIR_CAPITAL / 2
        pnl                     = sign * alloc * ret_A + (-sign) * alloc * ret_B
        blotter['gross_return'] = pnl / PAIR_CAPITAL
        blotter['cost']         = 2 * COST_BPS / 10_000
        blotter['net_return']   = blotter['gross_return'] - blotter['cost']

    # Build ledger
    ledger_rows    = []
    running_equity = float(PAIR_CAPITAL)
    prev_A = None
    prev_B = None

    trade_lookup = {}
    if len(blotter) > 0:
        for _, t in blotter.iterrows():
            for d in backtest_prices.loc[t['open_date']:t['close_date']].index:
                trade_lookup[d] = t

    for i, (date, row) in enumerate(backtest_prices.iterrows()):
        t         = trade_lookup.get(date)
        pos       = 'flat'
        daily_pnl = 0.0

        if t is not None:
            pos  = t['direction']
            sign = 1 if pos == 'long_spread' else -1

            if prev_A is not None and date != t['open_date']:
                dr_A           = row[sym_a] / prev_A - 1
                dr_B           = row[sym_b] / prev_B - 1
                daily_ret      = sign * 0.5 * dr_A + (-sign) * 0.5 * dr_B
                daily_pnl      = running_equity * daily_ret
                running_equity += daily_pnl

            if date == t['open_date'] or date == t['close_date']:
                running_equity -= PAIR_CAPITAL * (COST_BPS / 10_000)

        prev_A = row[sym_a]
        prev_B = row[sym_b]

        ledger_rows.append({
            'date':            date,
            'pair':            name,
            'position':        pos,
            'zscore':          round(zscores[i], 6),
            'spread':          round(spreads[i], 6),
            'hedge_ratio':     round(hrs_bt[i],  6),
            'price_A':         row[sym_a],
            'price_B':         row[sym_b],
            'daily_pnl':       round(daily_pnl, 2),
            'portfolio_value': round(running_equity, 2),
        })

    ledger = pd.DataFrame(ledger_rows)
    ledger['date'] = pd.to_datetime(ledger['date'])

    all_blotters.append(blotter)
    all_ledgers.append(ledger)
    print(f'{name}: {len(blotter)} trades')

Backtest: 294 days  (2025-03-03 to 2026-05-01)
SLB/HAL: 3 trades
PSX/VLO: 3 trades
XOM/CVX: 0 trades


## Section 6 — Blotter (trades.csv)

Combines all pair blotters into a single sorted trade log.
P&L uses dollar-neutral sizing ($16,667 per leg) with 5bps one-way cost.

In [69]:
trades = pd.concat([b for b in all_blotters if len(b) > 0], ignore_index=True)
trades = trades.sort_values('open_date').reset_index(drop=True)

trades.to_csv('trades.csv', index=False)
print(f'trades.csv written — {len(trades)} trades total')
# display(trades[['pair','open_date','close_date','direction',
#                 'entry_z','exit_z','exit_type','hold_days',
#                 'gross_return','net_return']])

from itables import show
import itables.options as opt
opt.lengthMenu = [10, 25, 50]

# Format columns manually before showing
trades_display = trades[['pair','open_date','close_date','direction',
                          'entry_z','exit_z','exit_type','hold_days',
                          'gross_return','net_return']].copy()

trades_display['gross_return'] = trades_display['gross_return'].map('{:+.3%}'.format)
trades_display['net_return']   = trades_display['net_return'].map('{:+.3%}'.format)
trades_display['entry_z']      = trades_display['entry_z'].map('{:.3f}'.format)
trades_display['exit_z']       = trades_display['exit_z'].map('{:.3f}'.format)

print('Interactive Blotter:')
show(trades_display)

import plotly.graph_objects as go

# Blotter table
fig_blotter = go.Figure(data=[go.Table(
    header=dict(
        values=list(trades_display.columns),
        fill_color='#f5f5f7',
        font=dict(size=12, color='#1d1d1f', family='-apple-system'),
        align='left', height=36
    ),
    cells=dict(
        values=[trades_display[c] for c in trades_display.columns],
        fill_color=[['white', '#fafafa'] * len(trades_display)],
        font=dict(size=11, color='#3a3a3c'),
        align='left', height=32
    )
)])
fig_blotter.update_layout(margin=dict(l=0, r=0, t=0, b=0), height=280)
fig_blotter.write_html('blotter_table.html')
print('blotter_table.html saved')

# Ledger table
fig_ledger = go.Figure(data=[go.Table(
    header=dict(
        values=list(ledger_display.columns),
        fill_color='#f5f5f7',
        font=dict(size=12, color='#1d1d1f', family='-apple-system'),
        align='left', height=36
    ),
    cells=dict(
        values=[ledger_display[c] for c in ledger_display.columns],
        fill_color=[['white', '#fafafa'] * len(ledger_display)],
        font=dict(size=11, color='#3a3a3c'),
        align='left', height=30
    )
)])
fig_ledger.update_layout(margin=dict(l=0, r=0, t=0, b=0), height=600)
fig_ledger.write_html('ledger_table.html')
print('ledger_table.html saved')

trades.csv written — 6 trades total
Interactive Blotter:


Loading ITables v2.7.3 from the internet... (need help?)


blotter_table.html saved
ledger_table.html saved


## Section 7 — Ledger (ledger.csv)

Combines per-pair daily ledgers into a portfolio-level ledger.
portfolio_value = sum of three pair equity curves.

In [70]:
# Combine three pair ledgers into one portfolio ledger
portfolio = all_ledgers[0][['date']].copy()
portfolio['portfolio_value'] = sum(l['portfolio_value'].values for l in all_ledgers)
portfolio['daily_pnl']       = sum(l['daily_pnl'].values       for l in all_ledgers)

for i, (_, _, name) in enumerate(PAIRS):
    short = name.replace('/', '_')
    portfolio[f'zscore_{short}']    = all_ledgers[i]['zscore'].values
    portfolio[f'position_{short}']  = all_ledgers[i]['position'].values
    portfolio[f'hedgert_{short}']   = all_ledgers[i]['hedge_ratio'].values
    portfolio[f'value_{short}']     = all_ledgers[i]['portfolio_value'].values

portfolio.to_csv('ledger.csv', index=False)
print(f'ledger.csv written — {len(portfolio)} days')
print(f'Final portfolio value: ${portfolio["portfolio_value"].iloc[-1]:,.2f}')
portfolio[['date','portfolio_value','daily_pnl',
           'zscore_SLB_HAL','zscore_PSX_VLO','zscore_XOM_CVX']].head(10)

from itables import show

ledger_display = portfolio[['date','portfolio_value','daily_pnl',
                            'position_SLB_HAL','position_PSX_VLO','position_XOM_CVX',
                            'zscore_SLB_HAL','zscore_PSX_VLO','zscore_XOM_CVX']].copy()

ledger_display['portfolio_value'] = ledger_display['portfolio_value'].map('${:,.2f}'.format)
ledger_display['daily_pnl']       = ledger_display['daily_pnl'].map('${:+,.2f}'.format)
ledger_display['zscore_SLB_HAL']  = ledger_display['zscore_SLB_HAL'].map('{:.3f}'.format)
ledger_display['zscore_PSX_VLO']  = ledger_display['zscore_PSX_VLO'].map('{:.3f}'.format)
ledger_display['zscore_XOM_CVX']  = ledger_display['zscore_XOM_CVX'].map('{:.3f}'.format)

print('Interactive Ledger:')
show(ledger_display)

import plotly.graph_objects as go

fig_ledger = go.Figure(data=[go.Table(
    header=dict(
        values=list(ledger_display.columns),
        fill_color='#f5f5f7',
        font=dict(size=12, color='#1d1d1f', family='-apple-system'),
        align='left', height=36
    ),
    cells=dict(
        values=[ledger_display[c] for c in ledger_display.columns],
        fill_color=[['white', '#fafafa'] * len(ledger_display)],
        font=dict(size=11, color='#3a3a3c'),
        align='left', height=30
    )
)])
fig_ledger.update_layout(margin=dict(l=0, r=0, t=0, b=0), height=600)
fig_ledger.write_html('ledger_table.html')
print('ledger_table.html saved')

ledger.csv written — 294 days
Final portfolio value: $98,564.59
Interactive Ledger:


Loading ITables v2.7.3 from the internet... (need help?)


ledger_table.html saved


## Section 8 — Performance Metrics

Reports per-pair and portfolio-level statistics including
Sharpe ratio, max drawdown, win rate, and expected return per trade.

In [71]:
def performance_summary(ledger, blotter, label=''):
    equity      = ledger['portfolio_value']
    returns     = equity.pct_change().dropna()
    rolling_max = equity.cummax()
    drawdown    = (equity - rolling_max) / rolling_max

    total_return = (equity.iloc[-1] / equity.iloc[0] - 1) * 100
    ann_return   = ((equity.iloc[-1] / equity.iloc[0]) ** (252 / len(equity)) - 1) * 100
    ann_vol      = returns.std() * np.sqrt(252) * 100
    sharpe       = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0
    max_dd       = drawdown.min() * 100

    if len(blotter) > 0:
        win_rate  = (blotter['net_return'] > 0).mean() * 100
        avg_hold  = blotter['hold_days'].mean()
        n_trades  = len(blotter)
        n_stop    = (blotter['exit_type'] == 'stop_loss').sum()
        n_timeout = (blotter['exit_type'] == 'end_of_backtest').sum()
        exp_ret   = blotter['net_return'].mean() * 100
    else:
        win_rate = avg_hold = n_trades = n_stop = n_timeout = exp_ret = 0

    return {
        'label':        label,
        'total_return': round(total_return, 2),
        'ann_return':   round(ann_return, 2),
        'ann_vol':      round(ann_vol, 2),
        'sharpe':       round(sharpe, 3),
        'max_dd':       round(max_dd, 2),
        'win_rate':     round(win_rate, 1),
        'avg_hold':     round(float(avg_hold), 1) if avg_hold else 0,
        'n_trades':     n_trades,
        'n_stop':       int(n_stop),
        'n_timeout':    int(n_timeout),
        'exp_ret':      round(exp_ret, 3),
    }

# Per-pair summary
print('Per-Pair Performance')
print('=' * 55)
for i, (_, _, name) in enumerate(PAIRS):
    s = performance_summary(all_ledgers[i], all_blotters[i], label=name)
    print(f'\n{name}')
    print(f'  Total return     : {s["total_return"]:+.2f}%')
    print(f'  Ann. return      : {s["ann_return"]:+.2f}%')
    print(f'  Sharpe ratio     : {s["sharpe"]:.3f}')
    print(f'  Max drawdown     : {s["max_dd"]:.2f}%')
    print(f'  Win rate         : {s["win_rate"]:.1f}%')
    print(f'  Avg hold (days)  : {s["avg_hold"]:.1f}')
    print(f'  Trades           : {s["n_trades"]}')
    print(f'  Stop-loss exits  : {s["n_stop"]}')
    print(f'  Exp. return/trade: {s["exp_ret"]:+.3f}%')

# Portfolio summary
print()
print('Portfolio (combined)')
print('=' * 55)
port_summary = performance_summary(portfolio, trades, label='Portfolio')
print(f'  Total return     : {port_summary["total_return"]:+.2f}%')
print(f'  Ann. return      : {port_summary["ann_return"]:+.2f}%')
print(f'  Ann. volatility  : {port_summary["ann_vol"]:.2f}%')
print(f'  Sharpe ratio     : {port_summary["sharpe"]:.3f}')
print(f'  Max drawdown     : {port_summary["max_dd"]:.2f}%')
print(f'  Win rate         : {port_summary["win_rate"]:.1f}%')
print(f'  Avg hold (days)  : {port_summary["avg_hold"]:.1f}')
print(f'  Total trades     : {port_summary["n_trades"]}')
print(f'  Stop-loss exits  : {port_summary["n_stop"]}')
print(f'  Exp. return/trade: {port_summary["exp_ret"]:+.3f}%')

Per-Pair Performance

SLB/HAL
  Total return     : -5.29%
  Ann. return      : -4.55%
  Sharpe ratio     : -0.501
  Max drawdown     : -10.08%
  Win rate         : 33.3%
  Avg hold (days)  : 25.3
  Trades           : 3
  Stop-loss exits  : 0
  Exp. return/trade: -1.657%

PSX/VLO
  Total return     : +0.99%
  Ann. return      : +0.84%
  Sharpe ratio     : 0.198
  Max drawdown     : -3.42%
  Win rate         : 66.7%
  Avg hold (days)  : 35.3
  Trades           : 3
  Stop-loss exits  : 0
  Exp. return/trade: +0.514%

XOM/CVX
  Total return     : +0.00%
  Ann. return      : +0.00%
  Sharpe ratio     : 0.000
  Max drawdown     : 0.00%
  Win rate         : 0.0%
  Avg hold (days)  : 0.0
  Trades           : 0
  Stop-loss exits  : 0
  Exp. return/trade: +0.000%

Portfolio (combined)
  Total return     : -1.44%
  Ann. return      : -1.23%
  Ann. volatility  : 3.38%
  Sharpe ratio     : -0.351
  Max drawdown     : -3.42%
  Win rate         : 50.0%
  Avg hold (days)  : 30.3
  Total trades     : 6

## Section 8b — Extended Statistics

Profit factor, average win/loss, consecutive loss streak,
and exit type breakdown by pair.

In [72]:
# Extended trade statistics
print('Extended Trade Statistics')
print('=' * 45)

if len(trades) > 0:
    winners = trades[trades['net_return'] > 0]
    losers  = trades[trades['net_return'] < 0]

    avg_win  = winners['net_return'].mean() * 100 if len(winners) > 0 else 0
    avg_loss = losers['net_return'].mean()  * 100 if len(losers)  > 0 else 0

    gross_profit = winners['net_return'].sum()
    gross_loss   = abs(losers['net_return'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

    # Max consecutive losses
    results = (trades['net_return'] > 0).astype(int).tolist()
    max_consec_loss = cur = 0
    for r in results:
        cur = 0 if r else cur + 1
        max_consec_loss = max(max_consec_loss, cur)

    print(f'  Avg win per trade    : +{avg_win:.3f}%')
    print(f'  Avg loss per trade   : {avg_loss:.3f}%')
    print(f'  Profit factor        : {profit_factor:.3f}')
    print(f'  Max consecutive loss : {max_consec_loss}')
    print()

    # Exit type breakdown
    print('Exit Type Breakdown')
    print('=' * 45)
    exit_counts = trades['exit_type'].value_counts()
    for etype, count in exit_counts.items():
        rate = count / len(trades) * 100
        print(f'  {etype:<20}: {count} trades ({rate:.0f}%)')

    print()

    # Per-pair breakdown
    print('Per-Pair Exit Breakdown')
    print('=' * 45)
    for _, _, name in PAIRS:
        pair_trades = trades[trades['pair'] == name]
        if len(pair_trades) == 0:
            print(f'  {name}: no trades')
            continue
        wins   = (pair_trades['net_return'] > 0).sum()
        losses = (pair_trades['net_return'] <= 0).sum()
        print(f'  {name}: {len(pair_trades)} trades  |  {wins} wins  {losses} losses  |  avg return {pair_trades["net_return"].mean()*100:+.3f}%')


Extended Trade Statistics
  Avg win per trade    : +1.182%
  Avg loss per trade   : -2.325%
  Profit factor        : 0.508
  Max consecutive loss : 2

Exit Type Breakdown
  mean_reversion      : 6 trades (100%)

Per-Pair Exit Breakdown
  SLB/HAL: 3 trades  |  1 wins  2 losses  |  avg return -1.657%
  PSX/VLO: 3 trades  |  2 wins  1 losses  |  avg return +0.514%
  XOM/CVX: no trades


## Section 8c - Alpha & Beta vs SPY

In [73]:
# Fetch SPY as benchmark
spy_raw = fetch_daily('SPY')
spy_bt  = spy_raw[BACKTEST_START:].copy()

# Use pct_change for both — consistent and avoids index length mismatch
spy_returns  = spy_bt['SPY'].pct_change().dropna()
port_returns = portfolio.set_index('date')['portfolio_value'].pct_change().dropna()

# Ensure both indices are datetime
spy_returns.index  = pd.to_datetime(spy_returns.index)
port_returns.index = pd.to_datetime(port_returns.index)

# Align on common trading days
common_dates = port_returns.index.intersection(spy_returns.index)
print(f'Common dates: {len(common_dates)}')

port_ret_aligned = port_returns.loc[common_dates]
spy_ret_aligned  = spy_returns.loc[common_dates]

# OLS regression: portfolio return = alpha + beta * SPY return
model = OLS(port_ret_aligned, add_constant(spy_ret_aligned)).fit()
beta  = model.params.iloc[1]
alpha = model.params.iloc[0] * 252  # annualized

print()
print('Alpha & Beta vs SPY')
print('=' * 40)
print(f'  Alpha (annualized) : {alpha:+.4f}  ({alpha*100:+.2f}%)')
print(f'  Beta               : {beta:.4f}')
print(f'  R²                 : {model.rsquared:.4f}')
print()
print('Beta near 0 = market neutral (pairs trading goal).')
print('Alpha > 0   = outperformance vs SPY on risk-adjusted basis.')

Common dates: 293

Alpha & Beta vs SPY
  Alpha (annualized) : -0.0157  (-1.57%)
  Beta               : 0.0192
  R²                 : 0.0115

Beta near 0 = market neutral (pairs trading goal).
Alpha > 0   = outperformance vs SPY on risk-adjusted basis.


## Section 9 — Plots

Four-panel chart: equity curves, drawdown, z-scores with trade markers,
and per-trade net returns. Plus rolling hedge ratio evolution for each pair.

In [74]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = ['#2196F3', '#FF9800', '#4CAF50']

# Chart 1: Equity curves + Drawdown + Z-scores + Trade returns
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    subplot_titles=['Equity Curves', 'Portfolio Drawdown (%)',
                    'Z-scores — All Pairs', 'Net Return per Trade (%)'],
    row_heights=[0.35, 0.2, 0.3, 0.15],
    vertical_spacing=0.06
)

# Row 1: Equity curves
fig.add_trace(go.Scatter(
    x=portfolio['date'], y=portfolio['portfolio_value'],
    name='Portfolio', line=dict(color='black', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=portfolio['date'], y=[100_000]*len(portfolio),
    name='Initial capital', line=dict(color='gray', dash='dash', width=1),
    showlegend=True
), row=1, col=1)

for i, (_, _, name) in enumerate(PAIRS):
    col = f'value_{name.replace("/","_")}'
    fig.add_trace(go.Scatter(
        x=portfolio['date'], y=portfolio[col],
        name=name, line=dict(color=colors[i], width=1),
        opacity=0.7
    ), row=1, col=1)

# Row 2: Drawdown
eq          = portfolio['portfolio_value']
rolling_max = eq.cummax()
drawdown    = (eq - rolling_max) / rolling_max * 100

fig.add_trace(go.Scatter(
    x=portfolio['date'], y=drawdown,
    fill='tozeroy', fillcolor='rgba(220,50,50,0.3)',
    line=dict(color='rgba(220,50,50,0.8)', width=1),
    name='Drawdown', showlegend=False
), row=2, col=1)

# Row 3: Z-scores
for i, (_, _, name) in enumerate(PAIRS):
    col = f'zscore_{name.replace("/","_")}'
    fig.add_trace(go.Scatter(
        x=portfolio['date'], y=portfolio[col],
        name=f'z {name}', line=dict(color=colors[i], width=1),
        opacity=0.85
    ), row=3, col=1)

for level, color, dash in [(2.0, 'red', 'dot'), (-2.0, 'red', 'dot'),
                             (4.0, 'darkred', 'dashdot'), (-4.0, 'darkred', 'dashdot'),
                             (0, 'black', 'dash')]:
    fig.add_hline(y=level, line=dict(color=color, dash=dash, width=1),
                  row=3, col=1)

# Trade entry/exit markers on z-score chart
for _, t in trades.iterrows():
    pair_idx = [n for _, _, n in PAIRS].index(t['pair'])
    fig.add_vline(x=t['open_date'], line=dict(
        color=colors[pair_idx], width=1, dash='dot'), opacity=0.4, row=3, col=1)
    fig.add_vline(x=t['close_date'], line=dict(
        color='gray', width=1, dash='dot'), opacity=0.3, row=3, col=1)

# Row 4: Trade returns bar chart
bar_colors = ['#2196F3' if r > 0 else '#F44336'
              for r in trades['net_return']]
fig.add_trace(go.Bar(
    x=[f'T{i+1} {row["pair"]}' for i, row in trades.iterrows()],
    y=trades['net_return'] * 100,
    marker_color=bar_colors,
    name='Net return', showlegend=False,
    text=[f'{r*100:+.2f}%' for r in trades['net_return']],
    textposition='outside'
), row=4, col=1)

fig.update_layout(
    height=1000,
    template='plotly_white',
    font=dict(family='-apple-system, BlinkMacSystemFont, SF Pro Display', size=12),
    legend=dict(orientation='v', x=1.01, y=1),
    hovermode='x unified',
    margin=dict(l=60, r=150, t=60, b=40)
)

fig.update_yaxes(title_text='Value ($)',  row=1, col=1)
fig.update_yaxes(title_text='%',          row=2, col=1)
fig.update_yaxes(title_text='Z-score',    row=3, col=1)
fig.update_yaxes(title_text='%',          row=4, col=1)

fig.show()

# Chart 2: Rolling hedge ratios
fig2 = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[f'{name} — Rolling Hedge Ratio (60-day window)'
                    for _, _, name in PAIRS],
    vertical_spacing=0.08
)

for i, (sym_a, sym_b, name) in enumerate(PAIRS):
    ledger  = all_ledgers[i]
    static  = validation[name]['hedge_ratio']

    fig2.add_trace(go.Scatter(
        x=ledger['date'], y=ledger['hedge_ratio'],
        name=name, line=dict(color=colors[i], width=1.5)
    ), row=i+1, col=1)

    fig2.add_hline(
        y=static,
        line=dict(color='gray', dash='dash', width=1),
        annotation_text=f'Static OLS β={static:.4f}',
        annotation_position='top right',
        row=i+1, col=1
    )

fig2.update_layout(
    height=700,
    template='plotly_white',
    font=dict(family='-apple-system, BlinkMacSystemFont, SF Pro Display', size=12),
    showlegend=False,
    hovermode='x unified',
    margin=dict(l=60, r=60, t=60, b=40)
)

for i in range(1, 4):
    fig2.update_yaxes(title_text='β', row=i, col=1)

fig2.show()

# Save interactive charts for website
fig.write_html('chart_interactive.html')
fig2.write_html('hedge_ratio_interactive.html')
print('Charts saved.')

Charts saved.


## Section 10 — Discussion

### Strategy Overview

We ran a pairs trading strategy across three energy sector pairs simultaneously:
SLB/HAL (oilfield services), PSX/VLO (refining), and XOM/CVX (integrated majors).
All three pairs are driven by the same macro environment — oil prices, energy demand,
and upstream capex cycles — but react differently in the short term, creating
tradeable spreads.

The key design choice was a 60-day rolling OLS hedge ratio instead of a static
formation-period estimate. This allows the strategy to adapt as the relative
valuation between the two companies shifts over time.

### Statistical Validation

| Pair | EG p-value | ADF p-value | Result |
|------|-----------|------------|--------|
| SLB/HAL | 0.061 | 0.040 | ADF pass at 5% |
| PSX/VLO | 0.140 | 0.048 | ADF pass at 5% |
| XOM/CVX | 0.307 | 0.132 | Both marginal |

### Backtest Results

Portfolio total return: -1.44% over 294 trading days.
PSX/VLO was the best performer (+0.99%, 2/3 wins).
SLB/HAL was the main drag (-5.29%, 1/3 wins).
XOM/CVX generated no signals — z-score never crossed ±2.0.

All 6 trades exited via mean reversion with zero stop-loss triggers,
confirming the spread does eventually revert. The one-sided signal direction
(all long spread) suggests a systematic downward drift in the spread during
the backtest period.

### Key Risks

- **Sector concentration:** All three pairs are in energy — correlated drawdowns
- **Weak cointegration:** XOM/CVX failed statistical tests
- **Spread drift:** All trades long spread suggests directional bias
- **Small sample:** 6 trades over 294 days limits statistical inference

### How to Improve

- Add pairs from other sectors to reduce concentration risk
- Regime filter: avoid trading when VIX > 25
- Partial exits at z = 0.5 to lock in profits earlier
- Re-run formation annually to keep parameters fresh